# Studio 2 — Preview 2: data lengkap · eksperimen resolusi · TTA

Preview 1 memakai ConvNeXt-Tiny 224 px yang dilatih dengan **2.753 dari 2.964 gambar**. Analisis kesalahannya menunjukkan pola yang jelas: model paling sering keliru pada **keluarga gelas** (gelas plastik, gelas kertas, gelas styrofoam). Bentuknya sama; yang membedakan hanya **tekstur material**.

Notebook ini menguji **satu hipotesis** dengan **satu perubahan**:

> Resolusi input lebih besar (288 px) memberi model lebih banyak detail tekstur, sehingga keluarga gelas lebih mudah dibedakan.

| Bagian | Isi |
|---|---|
| **0 · Data** | Cek kelengkapan terhadap `train_sources.csv`. Notebook **berhenti** bila ada gambar yang hilang |
| **A · Eksperimen** | 224 px (acuan) vs 288 px pada split `my_val` yang sama. Tiap run disimpan ke Drive, jadi bisa dilanjutkan bila Colab terputus |
| **A2 · TTA** | Uji *test-time augmentation* flip horizontal pada model terpilih |
| **B · Final** | Latih ulang konfigurasi terpilih di **seluruh** data |
| **C · Paket** | `submission/` + `README.MD` siap commit, plus cek batas waktu evaluasi di CPU |

**Cara pakai:** Runtime → Change runtime type → **T4 GPU** → ubah sel **KONFIGURASI** → **Runtime → Run all**. Total ±45–60 menit.

⚠️ **Jangan commit notebook ini setelah dijalankan**: galeri kesalahan memuat gambar dataset. Yang masuk repo hanya versi bersih (tanpa output).

In [ ]:
!pip -q install -U "timm>=1.0"
import os, sys, json, shutil, subprocess, time, hashlib, string
from pathlib import Path
import torch, timm
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"torch {torch.__version__} | timm {timm.__version__} | GPU: {gpu or 'TIDAK ADA'}")
if gpu is None:
    print("PERINGATAN: tanpa GPU training sangat lambat. Runtime -> Change runtime type -> T4 GPU, lalu Run all lagi.")

CLASSES = ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam']


def run(cmd):
    # Jalankan perintah shell, tampilkan output langsung, dan HENTIKAN notebook bila gagal.
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONWARNINGS="ignore")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Perintah gagal (exit {p.returncode}). Baca pesan error di atas.")


def fmt(x, n=4):
    # 0.97215 -> '0,9722' (format desimal Indonesia untuk README)
    return f"{x:.{n}f}".replace(".", ",")

## KONFIGURASI — satu-satunya sel yang perlu diubah

**`DATA_SRC` harus menunjuk ke dataset yang LENGKAP (2.964 gambar).** Salinan yang dipakai di Preview 1 kehilangan 211 gambar. Cara paling aman: pakai folder asli dari asisten, jangan salinan hasil upload ulang.
1. Buka link dataset dari halaman kompetisi → klik kanan folder → **Organize → Add shortcut → My Drive**.
2. Isi `DATA_SRC` dengan path ke folder `train` di dalam shortcut itu, misalnya `/content/drive/MyDrive/<nama shortcut>/train`. Kalau ragu, setelah sel *Dataset* me-mount Drive, buka panel 📁 di kiri → klik kanan folder `train` → **Copy path**.

`EXPERIMENTS` berisi argumen `train.py` per eksperimen. Semua eksperimen memakai split `my_val` yang sama, sehingga skornya bisa dibandingkan langsung.

In [ ]:
# ===================== UBAH DI SINI =====================
DATA_SRC = '/content/drive/MyDrive/dataset_mahasiswa/train'   # folder train/ LENGKAP (2.964 gambar)
ALLOW_MISSING = False       # True = tetap jalan walau ada gambar hilang (TIDAK disarankan)

# Satu hipotesis -> satu perubahan. cnxt224 = acuan (setara Preview 1), cnxt288 = hipotesis.
EXPERIMENTS = {
    'cnxt224': '--backbone convnext_tiny.fb_in22k_ft_in1k --img-size 224',
    'cnxt288': '--backbone convnext_tiny.fb_in22k_ft_in1k --img-size 288',
}
EPOCHS = 15
LR     = 1e-4
BATCH  = 32
AUG    = 'strong'           # 'strong' atau 'light'
SEED   = 42
FINAL_EXP = 'auto'          # 'auto' = rata-rata Macro-F1 3 epoch terakhir tertinggi; atau nama, mis. 'cnxt288'
USE_TTA   = 'auto'          # 'auto' = aktif bila tidak menurunkan Macro-F1 my_val; atau True / False
SAVE_TO_DRIVE = '/content/drive/MyDrive/studio2_runs'   # tiap run disimpan di sini -> bisa dilanjutkan

# --- untuk README yang dibuat otomatis di Bagian C ---
COMMIT_PREVIEW1 = 'f88e1ad'   # commit terakhir sebelum tenggat Preview 1
SKOR_PREVIEW1   = '0,9595'    # dari Google Sheets (akurasi 0,9609)
# ========================================================
WORK = '/content/work' if IN_COLAB else os.path.abspath('work')
COMMON = f'--epochs {EPOCHS} --lr {LR} --batch-size {BATCH} --aug {AUG} --seed {SEED}'
print('kerja di', WORK)

## 0. Dataset dan cek kelengkapan

Data disalin dari Drive ke disk lokal Colab (jauh lebih cepat dibaca saat training), lalu **dicocokkan nama per nama** dengan `train_sources.csv`. Kalau ada gambar yang hilang, notebook berhenti dan menulis daftarnya ke `gambar_hilang.csv`. Tujuannya supaya masalah di Preview 1 (211 gambar hilang tanpa ketahuan) tidak terulang.

In [ ]:
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(DATA_SRC):
    raise FileNotFoundError(
        f'DATA_SRC tidak ditemukan: {DATA_SRC}\n'
        '-> cek path di panel file kiri (drive/MyDrive/...), lalu ubah sel KONFIGURASI.\n'
        '-> baru membuat / mengganti nama shortcut di Drive? Runtime -> Disconnect and delete runtime, lalu Run all.')
src_files = {p.relative_to(DATA_SRC).as_posix() for p in Path(DATA_SRC).glob('*/*.jpg')}
print(f'DATA_SRC: {DATA_SRC}  ->  {len(src_files)} gambar')

os.makedirs(os.path.join(WORK, 'submission'), exist_ok=True)
os.chdir(WORK)
DATA = os.path.join(WORK, 'data', 'train')
if os.path.isdir(DATA):
    # Bandingkan ISI, bukan sekadar nama path: salinan lokal harus sama persis dengan DATA_SRC.
    local = {p.relative_to(DATA).as_posix() for p in Path(DATA).glob('*/*.jpg')}
    if local != src_files:
        print(f'salinan lokal ({len(local)} gambar) berbeda dari DATA_SRC -> salinan lama dan split my_val dihapus')
        for d in ('data', 'my_train', 'my_val'):
            shutil.rmtree(d, ignore_errors=True)
if not os.path.isdir(DATA):
    t0 = time.time()
    shutil.copytree(DATA_SRC, DATA)
    print(f'disalin dalam {time.time() - t0:.0f} s')

EXPECTED = {'botol_plastik': 541, 'gelas_plastik': 572, 'kertas_kardus': 395, 'saset_kemasan': 758, 'styrofoam': 698}
present = {p.relative_to(DATA).as_posix() for p in Path(DATA).glob('*/*.jpg')}
SOURCES_CSV = os.path.join(os.path.dirname(DATA_SRC.rstrip('/')), 'train_sources.csv')
src = pd.read_csv(SOURCES_CSV) if os.path.exists(SOURCES_CSV) else None
if src is None:
    print('train_sources.csv tidak ditemukan di samping folder train/ -> hanya cek jumlah per kelas')

rows = []
for c in CLASSES:
    n_exp = int((src.label == c).sum()) if src is not None else EXPECTED[c]
    n_have = sum(1 for f in present if f.startswith(c + '/'))
    rows.append((c, n_exp, n_have, max(0, n_exp - n_have)))
tab = pd.DataFrame(rows, columns=['kelas', 'seharusnya', 'ada', 'hilang']).set_index('kelas')
tab.loc['TOTAL'] = tab.sum()
display(tab)

missing = sorted(set(src.file) - present) if src is not None else []
extra = sorted(present - set(src.file)) if src is not None else []
if extra:
    print(f'{len(extra)} file tidak tercatat di train_sources.csv (contoh: {extra[:3]})')
n_missing = int(tab.loc['TOTAL', 'hilang'])
N_TOTAL = len(present)
if n_missing:
    if missing:
        pd.Series(missing, name='file').to_csv('gambar_hilang.csv', index=False)
        print(f'daftar lengkap: {WORK}/gambar_hilang.csv  | contoh: {missing[:5]}')
    msg = (f'{n_missing} gambar hilang dari DATA_SRC. Arahkan DATA_SRC ke folder dataset asli dari asisten '
           '(Add shortcut), atau set ALLOW_MISSING = True bila memang disengaja.')
    if not ALLOW_MISSING:
        raise RuntimeError(msg)
    print('PERINGATAN:', msg)
else:
    print(f'Dataset LENGKAP: {N_TOTAL} gambar ✓')

## 1. Kode

Tiga sel berikut menulis file yang **sama persis** dengan yang ada di repo. Perubahan dibanding Preview 1:
- `submission/model.py`: dukungan TTA flip lewat kunci `tta_hflip` di `config.json`. Config lama tanpa kunci ini tetap berjalan seperti biasa.
- `train.py`: flag `--tta-hflip`, dan peringatan scheduler yang tidak relevan disembunyikan.

In [ ]:
%%writefile evaluate.py
"""Skrip evaluasi resmi Studio 2 — Klasifikasi Sampah (IF25-40401).

Skrip INI JUGA yang dipakai asisten dosen untuk menghitung skor pada validation set
tersembunyi. Pastikan submission kelompok Anda lolos skrip ini sebelum tenggat.

Pemakaian (uji pada split validasi buatan Anda sendiri dari data train):
    python evaluate.py --submission ./submission --data ./my_val

    --submission  folder berisi model.py dan model.pth
    --data        folder berstruktur <data>/<nama_kelas>/*.jpg
    --out         (opsional) simpan hasil ke file JSON

Kontrak submission (folder submission/):
    model.py   wajib mendefinisikan:
                 get_model()     -> torch.nn.Module (arsitektur saja, TANPA mengunduh bobot)
                 get_transform() -> callable: PIL.Image (RGB) -> torch.Tensor [3, H, W]
    model.pth  state_dict hasil torch.save(model.state_dict(), "model.pth")
               Output model: logits berukuran [batch, 5] dengan urutan kelas = CLASSES di bawah.
"""
import argparse
import importlib.util
import json
import sys
import time
from pathlib import Path

import torch
from PIL import Image

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def load_submission(sub_dir, device):
    sub_dir = Path(sub_dir).resolve()
    spec = importlib.util.spec_from_file_location("submission_model", sub_dir / "model.py")
    mod = importlib.util.module_from_spec(spec)
    sys.path.insert(0, str(sub_dir))  # izinkan model.py meng-import file pendukung di folder yang sama
    spec.loader.exec_module(mod)
    for fn in ("get_model", "get_transform"):
        if not hasattr(mod, fn):
            raise SystemExit(f"[GAGAL] model.py tidak mendefinisikan {fn}()")
    model = mod.get_model()
    state = torch.load(sub_dir / "model.pth", map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model, mod.get_transform()


def list_images(data_dir):
    items = []
    for ci, cls in enumerate(CLASSES):
        d = Path(data_dir) / cls
        if not d.is_dir():
            raise SystemExit(f"[GAGAL] folder kelas tidak ditemukan: {d}")
        items += [(p, ci) for p in sorted(d.iterdir()) if p.suffix.lower() in EXTS]
    return items


def metrics(y_true, y_pred, k=len(CLASSES)):
    cm = [[0] * k for _ in range(k)]
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    f1s = []
    for c in range(k):
        tp = cm[c][c]
        fp = sum(cm[r][c] for r in range(k)) - tp
        fn = sum(cm[c]) - tp
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    acc = sum(cm[c][c] for c in range(k)) / max(1, len(y_true))
    return {"macro_f1": sum(f1s) / k, "accuracy": acc,
            "per_class_f1": dict(zip(CLASSES, f1s)), "confusion_matrix": cm}


@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--submission", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--out")
    a = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    model, tf = load_submission(a.submission, device)
    items = list_images(a.data)
    print(f"device={device}  gambar={len(items)}")

    y_true, y_pred, t0 = [], [], time.time()
    for i in range(0, len(items), a.batch_size):
        chunk = items[i:i + a.batch_size]
        x = torch.stack([tf(Image.open(p).convert("RGB")) for p, _ in chunk]).to(device)
        logits = model(x)
        if logits.ndim != 2 or logits.shape[1] != len(CLASSES):
            raise SystemExit(f"[GAGAL] output model harus [batch, {len(CLASSES)}], didapat {tuple(logits.shape)}")
        y_pred += logits.argmax(1).cpu().tolist()
        y_true += [c for _, c in chunk]

    m = metrics(y_true, y_pred)
    m["n_images"] = len(items)
    m["seconds"] = round(time.time() - t0, 1)
    print(f"\nMacro-F1 : {m['macro_f1']:.4f}\nAkurasi  : {m['accuracy']:.4f}")
    for c, f in m["per_class_f1"].items():
        print(f"  F1 {c:<15} {f:.4f}")
    print("Confusion matrix (baris = label asli, kolom = prediksi):")
    for c, row in zip(CLASSES, m["confusion_matrix"]):
        print(f"  {c:<15} {row}")
    if a.out:
        Path(a.out).write_text(json.dumps(m, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile submission/model.py
"""Submission Studio 2 — model generik berbasis timm.

Arsitektur & preprocessing TIDAK ditulis langsung di sini, tetapi dibaca dari
`config.json` di folder yang sama (dibuat otomatis oleh train.py). Alasannya:
konfigurasi selalu "ikut" bersama bobotnya, sehingga model.py, config.json,
dan model.pth tidak mungkin saling tidak cocok tanpa ketahuan.

Isi folder submission/ yang wajib di-commit:
    model.py      (file ini — tidak perlu diubah)
    config.json   (backbone, ukuran input, mean/std, tta_hflip)
    model.pth     (state_dict)

Kontrak evaluate.py:
    get_model()     -> nn.Module, output logits [batch, 5], TANPA mengunduh bobot
    get_transform() -> PIL.Image (RGB) -> Tensor [3, H, W]
Urutan kelas: 0 botol_plastik, 1 gelas_plastik, 2 kertas_kardus, 3 saset_kemasan, 4 styrofoam
"""
import json
from pathlib import Path

import timm
import torch
from torchvision import transforms

NUM_CLASSES = 5
CONFIG_PATH = Path(__file__).resolve().parent / "config.json"


def load_config(path=CONFIG_PATH):
    if not Path(path).exists():
        raise FileNotFoundError(f"config.json tidak ditemukan di {Path(path).parent} — "
                                "salin config.json dari folder run bersama model.pth.")
    return json.loads(Path(path).read_text())


def build_model(backbone, pretrained=False):
    """Dipakai train.py (pretrained=True) dan get_model() (pretrained=False).
    Fungsi yang sama -> nama key state_dict pasti identik."""
    return timm.create_model(backbone, pretrained=pretrained, num_classes=NUM_CLASSES)


def build_eval_transform(img_size, crop_pct, mean, std):
    return transforms.Compose([
        transforms.Resize(int(round(img_size / crop_pct))),  # sisi pendek
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def add_hflip_tta(model):
    """Test-time augmentation: rata-rata logits gambar asli dan cerminan horizontalnya.

    Hanya `forward` yang dibungkus — tidak ada layer baru, jadi nama key state_dict
    tetap sama dan load_state_dict(strict=True) di evaluate.py tetap lolos.
    Masuk akal karena model dilatih dengan RandomHorizontalFlip: kedua versi gambar
    sama-sama "normal" bagi model, dan merata-ratakannya mengurangi variansi prediksi.
    """
    base_forward = model.forward

    def forward(x):
        return (base_forward(x) + base_forward(torch.flip(x, dims=[3]))) / 2

    model.forward = forward
    return model


def get_model():
    cfg = load_config()
    model = build_model(cfg["backbone"], pretrained=False)
    if cfg.get("tta_hflip", False):   # config lama tanpa kunci ini -> tanpa TTA
        model = add_hflip_tta(model)
    return model


def get_transform():
    cfg = load_config()
    return build_eval_transform(cfg["img_size"], cfg["crop_pct"], cfg["mean"], cfg["std"])

In [ ]:
%%writefile train.py
"""Training Studio 2 — fine-tuning backbone timm, checkpoint dipilih berdasarkan Macro-F1.

Jalankan dari root repo:
    python train.py --data ../data/train --backbone convnext_tiny.fb_in22k_ft_in1k --epochs 15 --name cnxt_e15
    python evaluate.py --submission runs/cnxt_e15/submission --data ./my_val

Setiap run menghasilkan runs/<name>/:
    submission/          model.py + config.json + model.pth -> folder submission lengkap
    log.csv              metrik per epoch (loss, Macro-F1, akurasi, F1 per kelas)
    val_predictions.csv  prediksi checkpoint terbaik di my_val, diurutkan dari yang paling "salah"
    args.json            semua argumen run (reproduksibilitas)

Mode --full: latih di my_train + my_val dengan jumlah epoch tetap (untuk model Final).
--tta-hflip: config.json submission menyalakan TTA flip horizontal (lihat submission/model.py).

Perbedaan utama dari train_baseline.py:
  1. Checkpoint dipilih berdasarkan Macro-F1 (metrik penilaian), bukan akurasi.
  2. Metrik & jalur load memakai fungsi dari evaluate.py -> angka identik dengan penilai.
  3. Mean/std/ukuran input diambil dari pretrained_cfg backbone, disimpan di config.json.
  4. Warmup + cosine per-step, LR head lebih besar dari backbone, AMP di GPU.
  5. Bobot > 95 MB otomatis disimpan fp16 agar muat di GitHub (< 100 MB).
"""
import argparse
import contextlib
import csv
import json
import math
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader
from PIL import Image
from torchvision import datasets, transforms

import evaluate                       # metrik + jalur load yang SAMA dengan penilai
from submission import model as sub   # build_model / build_eval_transform

# GradScaler melewati optimizer.step() di batch pertama (cek overflow) -> peringatan scheduler tidak relevan.
warnings.filterwarnings("ignore", message=r".*lr_scheduler\.step\(\).*")

CLASSES = evaluate.CLASSES
FP16_EXPORT_MB = 95


# ----------------------------------------------------------------------------- data
def make_local_split(src, dst, frac, seed):
    """Sama persis dengan train_baseline.py -> split my_val identik bila seed/frac sama.
    Hanya dibuat sekali; kalau my_val/ sudah ada, split lama dipakai ulang."""
    dst_tr, dst_va = Path(dst) / "my_train", Path(dst) / "my_val"
    if dst_va.exists():
        return dst_tr, dst_va
    rng = random.Random(seed)
    for cls_dir in sorted(Path(src).iterdir()):
        if not cls_dir.is_dir():
            continue
        files = sorted(cls_dir.glob("*.jpg"))
        rng.shuffle(files)
        n_va = int(len(files) * frac)
        for i, f in enumerate(files):
            out = (dst_va if i < n_va else dst_tr) / cls_dir.name
            out.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, out / f.name)
    return dst_tr, dst_va


def build_train_transform(cfg, aug):
    size = cfg["img_size"]
    to_tensor = [transforms.ToTensor(), transforms.Normalize(cfg["mean"], cfg["std"])]
    if aug == "light":  # setara baseline
        return transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2),
            *to_tensor,
        ])
    return transforms.Compose([  # "strong"
        transforms.RandomResizedCrop(size, scale=(0.4, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        *to_tensor,
        transforms.RandomErasing(p=0.25),
    ])


# ----------------------------------------------------------------------------- utils
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def model_config(model, img_size):
    pc = model.pretrained_cfg
    return {
        "backbone": None,  # diisi pemanggil
        "img_size": img_size or pc["input_size"][-1],
        "crop_pct": pc.get("crop_pct") or 0.875,
        "mean": [float(v) for v in pc["mean"]],
        "std": [float(v) for v in pc["std"]],
        "timm_version": timm.__version__,
        "torch_version": torch.__version__,
    }


@torch.no_grad()
def predict(model, dl, device):
    model.eval()
    probs, ys, loss_sum = [], [], 0.0
    for x, y in dl:
        logits = model(x.to(device)).float()
        loss_sum += F.cross_entropy(logits, y.to(device), reduction="sum").item()
        probs.append(logits.softmax(1).cpu())
        ys.append(y)
    p, y = torch.cat(probs), torch.cat(ys)
    return y.tolist(), p, loss_sum / len(y)


def export_submission(state, cfg, out_dir):
    """Tulis folder submission lengkap: model.py + config.json + model.pth."""
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(sub.__file__), out_dir / "model.py")
    (out_dir / "config.json").write_text(json.dumps(cfg, indent=2))
    mb = sum(v.numel() * v.element_size() for v in state.values()) / 1e6
    if mb > FP16_EXPORT_MB:
        state = {k: (v.half() if v.is_floating_point() else v) for k, v in state.items()}
        print(f"bobot {mb:.0f} MB > {FP16_EXPORT_MB} MB -> disimpan fp16")
    torch.save(state, out_dir / "model.pth")
    return (out_dir / "model.pth").stat().st_size / 1e6


@torch.no_grad()
def official_check(sub_dir, data_dir, device):
    """Muat submission PERSIS seperti evaluate.py (strict=True, weights_only=True)."""
    model, tf = evaluate.load_submission(sub_dir, device)
    items = evaluate.list_images(data_dir)
    y_true, y_pred = [], []
    for i in range(0, len(items), 32):
        chunk = items[i:i + 32]
        x = torch.stack([tf(Image.open(p).convert("RGB")) for p, _ in chunk]).to(device)
        y_pred += model(x).argmax(1).cpu().tolist()
        y_true += [c for _, c in chunk]
    return evaluate.metrics(y_true, y_pred)


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True, help="folder train/ asli dari dataset")
    ap.add_argument("--split-dir", default=".", help="lokasi my_train/ dan my_val/")
    ap.add_argument("--backbone", default="convnext_tiny.fb_in22k_ft_in1k", help="nama model timm")
    ap.add_argument("--img-size", type=int, default=None, help="default: ukuran pretrained backbone")
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4, help="LR backbone")
    ap.add_argument("--head-lr-mult", type=float, default=10.0, help="LR head = lr x nilai ini")
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup-epochs", type=float, default=1.0)
    ap.add_argument("--label-smoothing", type=float, default=0.1)
    ap.add_argument("--aug", choices=["light", "strong"], default="strong")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--name", default=None, help="nama folder di runs/")
    ap.add_argument("--full", action="store_true", help="latih di my_train + my_val (model Final)")
    ap.add_argument("--no-pretrained", action="store_true", help="tanpa bobot ImageNet (uji/scratch)")
    ap.add_argument("--tta-hflip", action="store_true",
                    help="aktifkan TTA flip horizontal di config.json submission (berlaku saat evaluasi)")
    a = ap.parse_args()

    seed_everything(a.seed)
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    name = a.name or f"{a.backbone.split('.')[0]}_{time.strftime('%m%d-%H%M')}"
    run = Path("runs") / name
    run.mkdir(parents=True, exist_ok=True)
    (run / "args.json").write_text(json.dumps(vars(a), indent=2))

    # ---- model + konfigurasi preprocessing (sumber kebenaran tunggal)
    model = sub.build_model(a.backbone, pretrained=not a.no_pretrained)
    cfg = model_config(model, a.img_size)
    cfg["backbone"] = a.backbone
    cfg["tta_hflip"] = a.tta_hflip
    model.to(device)
    print(f"device={device}  backbone={a.backbone}  input={cfg['img_size']}  mean={cfg['mean']}")

    # ---- data
    tr_dir, va_dir = make_local_split(a.data, a.split_dir, 0.15, 42)
    train_tf = build_train_transform(cfg, a.aug)
    eval_tf = sub.build_eval_transform(cfg["img_size"], cfg["crop_pct"], cfg["mean"], cfg["std"])
    tr_ds = datasets.ImageFolder(tr_dir, train_tf)
    va_ds = datasets.ImageFolder(va_dir, eval_tf)
    assert tr_ds.classes == CLASSES == va_ds.classes, f"urutan kelas salah: {tr_ds.classes}"
    if a.full:
        tr_ds = ConcatDataset([tr_ds, datasets.ImageFolder(va_dir, train_tf)])
    print(f"train={len(tr_ds)}  val={len(va_ds)}  mode={'FULL (tanpa validasi)' if a.full else 'normal'}")

    kw = dict(num_workers=a.workers, pin_memory=device == "cuda", persistent_workers=a.workers > 0)
    tr_dl = DataLoader(tr_ds, a.batch_size, shuffle=True, drop_last=True, **kw)
    va_dl = DataLoader(va_ds, a.batch_size, shuffle=False, **kw)

    # ---- optimizer: head belajar lebih cepat daripada backbone pretrained
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if id(p) not in head_ids], "lr": a.lr},
        {"params": [p for p in model.parameters() if id(p) in head_ids], "lr": a.lr * a.head_lr_mult},
    ], weight_decay=a.wd)
    total = a.epochs * len(tr_dl)
    warm = max(1, int(a.warmup_epochs * len(tr_dl)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    loss_fn = nn.CrossEntropyLoss(label_smoothing=a.label_smoothing)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    amp_ctx = (lambda: torch.autocast("cuda", dtype=torch.float16)) if use_amp else contextlib.nullcontext

    # ---- training loop
    header = ["epoch", "lr", "train_loss", "val_loss", "macro_f1", "acc"] + [f"f1_{c}" for c in CLASSES] + ["sec"]
    log_f = open(run / "log.csv", "w", newline="")
    log = csv.writer(log_f)
    log.writerow(header)
    best_f1, best_loss, best_ep, best_state, best_probs = -1.0, float("inf"), 0, None, None

    for ep in range(1, a.epochs + 1):
        t0 = time.time()
        model.train()
        loss_sum, n = 0.0, 0
        for x, y in tr_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with amp_ctx():
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.item() * len(y)
            n += len(y)

        y_true, probs, val_loss = predict(model, va_dl, device)
        m = evaluate.metrics(y_true, probs.argmax(1).tolist())
        sec = time.time() - t0
        log.writerow([ep, f"{opt.param_groups[0]['lr']:.2e}", f"{loss_sum / n:.4f}", f"{val_loss:.4f}",
                      f"{m['macro_f1']:.4f}", f"{m['accuracy']:.4f}"]
                     + [f"{m['per_class_f1'][c]:.4f}" for c in CLASSES] + [f"{sec:.0f}"])
        log_f.flush()

        # Macro-F1 lebih tinggi menang; bila seri, pilih val_loss lebih rendah.
        # Mode --full tidak punya validasi yang jujur -> selalu ambil epoch terakhir.
        improved = a.full or m["macro_f1"] > best_f1 or (m["macro_f1"] == best_f1 and val_loss < best_loss)
        if improved:
            best_f1, best_loss, best_ep, best_probs = m["macro_f1"], val_loss, ep, probs
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"epoch {ep:2d}/{a.epochs}  train_loss={loss_sum / n:.4f}  val_loss={val_loss:.4f}  "
              f"macroF1={m['macro_f1']:.4f}  acc={m['accuracy']:.4f}  {sec:.0f}s {'*' if improved else ''}")
    log_f.close()

    # ---- prediksi checkpoint terbaik -> bahan analisis error & label noise
    with open(run / "val_predictions.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "label", "pred", "p_label", "p_pred", "benar"])
        rows = []
        for (path, lab), p in zip(va_ds.samples, best_probs):
            pr = int(p.argmax())
            rows.append([Path(path).relative_to(va_dir).as_posix(), CLASSES[lab], CLASSES[pr],
                         round(float(p[lab]), 4), round(float(p[pr]), 4), int(pr == lab)])
        w.writerows(sorted(rows, key=lambda r: r[3]))  # paling "yakin salah" di atas

    # ---- ekspor + cek lewat jalur evaluate.py
    sub_dir = run / "submission"
    size_mb = export_submission(best_state, cfg, sub_dir)
    check = official_check(sub_dir, va_dir, device)
    (run / "best.json").write_text(json.dumps({
        "epoch": best_ep, "epochs": a.epochs, "full": a.full,
        "macro_f1_train_loop": round(best_f1, 4), "macro_f1_evaluate": round(check["macro_f1"], 4),
        "accuracy_evaluate": round(check["accuracy"], 4), "per_class_f1": check["per_class_f1"],
        "confusion_matrix": check["confusion_matrix"], "model_mb": round(size_mb, 1),
        "img_size": cfg["img_size"], "tta_hflip": a.tta_hflip, "n_train": len(tr_ds),
    }, indent=2))
    print("\n" + "=" * 70)
    print(f"run              : {run}")
    print(f"checkpoint       : epoch {best_ep}  (Macro-F1 my_val saat training = {best_f1:.4f})")
    print(f"cek evaluate.py  : Macro-F1 = {check['macro_f1']:.4f}  akurasi = {check['accuracy']:.4f}")
    print(f"model.pth        : {size_mb:.1f} MB  |  timm {timm.__version__}")
    if a.full:
        print("CATATAN: mode --full -> my_val ikut dilatih, skor di atas hanya cek 'bisa jalan'.")
    print(f"\nPromosikan ke submission/:  copy {sub_dir}/* -> submission/")
    print("=" * 70)


if __name__ == "__main__":
    main()

## 2. Split validasi lokal

`my_val` adalah 15% per kelas (seed 42). Karena datanya sekarang lengkap, isi `my_val` **berbeda** dari Preview 1. Akibatnya skor di sini **tidak bisa dibandingkan langsung dengan 0,9722**, dan itulah alasan eksperimen acuan 224 px dijalankan ulang.

*Sidik jari* split (hash daftar file `my_val`) dipakai sebagai nama folder penyimpanan di Drive. Run yang dipulihkan dari Drive dijamin memakai split yang sama.

In [ ]:
Path('make_split.py').write_text(
    "import sys\nfrom train import make_local_split\nmake_local_split(sys.argv[1], '.', 0.15, 42)\n")
run(f'python -u make_split.py "{DATA}"')

val_files = sorted(p.relative_to('my_val').as_posix() for p in Path('my_val').glob('*/*.jpg'))
FP = hashlib.md5('\n'.join(val_files).encode()).hexdigest()[:8]
RUNS_DRIVE = os.path.join(SAVE_TO_DRIVE, f'runs_{FP}') if SAVE_TO_DRIVE else None
split = pd.DataFrame({
    'my_train': [len(list(Path('my_train', c).glob('*.jpg'))) for c in CLASSES],
    'my_val': [len(list(Path('my_val', c).glob('*.jpg'))) for c in CLASSES],
}, index=CLASSES)
split.loc['TOTAL'] = split.sum()
display(split)
print('sidik jari split:', FP, '| simpan run di:', RUNS_DRIVE)

---
# Bagian A — Eksperimen

Setiap eksperimen dilatih pada `my_train` dan dinilai di `my_val` yang sama. Setelah selesai, hasilnya disalin ke Drive. Kalau Colab terputus, cukup **Run all** lagi: run yang sudah selesai dipulihkan dari Drive, tidak dilatih ulang.

Perkiraan waktu di T4: 224 px ±8 menit, 288 px ±13 menit.

In [ ]:
def restore_or_train(name, args):
    # Latih runs/<name>, KECUALI hasil dengan perintah yang sama sudah ada (di sesi ini atau di Drive).
    cmd = f'python -u train.py --data {DATA} --split-dir . --name {name} {COMMON} {args}'
    key = hashlib.md5(f'{COMMON} {args}'.encode()).hexdigest()[:6]
    run_dir = os.path.join('runs', name)
    drive_dir = os.path.join(RUNS_DRIVE, f'{name}_{key}') if RUNS_DRIVE else None

    def done(d):
        return (os.path.exists(os.path.join(d, 'best.json')) and os.path.exists(os.path.join(d, 'cmd.txt'))
                and open(os.path.join(d, 'cmd.txt')).read() == f'{COMMON} {args}')

    if done(run_dir):
        print(f'[{name}] sudah ada di sesi ini -> dilewati')
        return
    if drive_dir and done(drive_dir):
        shutil.rmtree(run_dir, ignore_errors=True)
        shutil.copytree(drive_dir, run_dir)
        print(f'[{name}] dipulihkan dari Drive ({drive_dir}) -> tidak dilatih ulang')
        return
    t0 = time.time()
    run(cmd)
    Path(run_dir, 'cmd.txt').write_text(f'{COMMON} {args}')
    print(f'[{name}] selesai dalam {(time.time() - t0) / 60:.1f} menit')
    if drive_dir:
        shutil.copytree(run_dir, drive_dir, dirs_exist_ok=True)
        print(f'[{name}] disimpan ke Drive: {drive_dir}')


for name, args in EXPERIMENTS.items():
    print(f'\n{"#" * 20} A_{name} {"#" * 20}')
    restore_or_train(f'A_{name}', args)

### Perbandingan eksperimen

Cara membaca:
- **`f1_rata3_akhir`** (rata-rata Macro-F1 3 epoch terakhir) dipakai untuk **memilih** eksperimen, bukan `f1_terbaik`. Nilai maksimum dari kurva yang berisik cenderung terlalu optimis (*selection bias*). Rata-rata beberapa epoch di ujung jadwal lebih stabil.
- **Selisih < 0,01** antara dua eksperimen masih dalam batas noise `my_val` (±440 gambar). Satu gambar `kertas_kardus` yang salah saja sudah menggeser Macro-F1 sekitar 0,003.
- **Kolom `gelas` / `kertas` / `styrofoam`**: di sinilah hipotesis diuji. Apakah 288 px memperbaiki keluarga gelas?

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # urutan warna tetap
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e5e4e0'


def style(ax, title, ylabel, xlabel='epoch'):
    ax.set_title(title, loc='left', color=INK, fontsize=11)
    ax.set_xlabel(xlabel, color=MUTED); ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED)


rows, logs, bests = [], {}, {}
for name in EXPERIMENTS:
    b = json.load(open(f'runs/A_{name}/best.json'))
    log = pd.read_csv(f'runs/A_{name}/log.csv')
    logs[name], bests[name] = log, b
    pc = b['per_class_f1']
    rows.append({'eksperimen': name, 'f1_terbaik': b['macro_f1_evaluate'], 'epoch_terbaik': b['epoch'],
                 'f1_rata3_akhir': log.macro_f1.tail(3).mean(), 'val_loss_akhir': log.val_loss.iloc[-1],
                 'gelas': pc['gelas_plastik'], 'kertas': pc['kertas_kardus'], 'styrofoam': pc['styrofoam'],
                 'menit': log.sec.sum() / 60})
summary = pd.DataFrame(rows).set_index('eksperimen')
display(summary.round(4))

CHOSEN = summary.f1_rata3_akhir.idxmax() if FINAL_EXP == 'auto' else FINAL_EXP
assert CHOSEN in EXPERIMENTS, f'FINAL_EXP tidak dikenal: {CHOSEN}'
ranked = summary.f1_rata3_akhir.sort_values(ascending=False)
gap = ranked.iloc[0] - ranked.iloc[1] if len(ranked) > 1 else float('nan')
print(f'Terpilih: {CHOSEN}  (rata-3-akhir {fmt(summary.loc[CHOSEN, "f1_rata3_akhir"])}'
      + (f', unggul {fmt(gap)} dari peringkat 2)' if len(ranked) > 1 else ')'))
if len(ranked) > 1 and gap < 0.01:
    print('Catatan: selisih < 0,01 -> masih dalam batas noise. Pilihan ini BELUM terbukti lebih baik; '
          'skor Preview 2 yang akan menjawabnya.')

print('\nKebingungan terbesar per eksperimen (label asli -> prediksi : jumlah):')
for name, b in bests.items():
    cm = b['confusion_matrix']
    pairs = sorted(((cm[i][j], CLASSES[i], CLASSES[j]) for i in range(5) for j in range(5) if i != j), reverse=True)
    print(f'  {name:10s}', ' | '.join(f'{t}->{p}: {n}' for n, t, p in pairs[:4] if n))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
for k, (name, log) in enumerate(logs.items()):
    axes[0].plot(log.epoch, log.macro_f1, color=SERIES[k], lw=2, marker='o', ms=3, label=name)
    axes[1].plot(log.epoch, log.val_loss, color=SERIES[k], lw=2, marker='o', ms=3, label=name)
style(axes[0], 'Macro-F1 my_val per epoch', 'Macro-F1')
style(axes[1], 'Loss validasi per epoch', 'cross-entropy')
for ax in axes[:2]:
    ax.xaxis.set_major_locator(MaxNLocator(integer=True)); ax.legend(frameon=False, labelcolor=MUTED)
width = 0.8 / len(bests)
for k, (name, b) in enumerate(bests.items()):
    xs = [i + (k - (len(bests) - 1) / 2) * width for i in range(len(CLASSES))]
    axes[2].bar(xs, [b['per_class_f1'][c] for c in CLASSES], width=width * 0.9, color=SERIES[k], label=name)
axes[2].set_xticks(range(len(CLASSES)), [c.replace('_', '\n') for c in CLASSES], fontsize=8)
lo = min(min(b['per_class_f1'].values()) for b in bests.values())
axes[2].set_ylim(max(0, lo - 0.05), 1.0)
style(axes[2], 'F1 per kelas (checkpoint terpilih)', 'F1', xlabel='')
axes[2].legend(frameon=False, labelcolor=MUTED, fontsize=8, ncol=len(bests),
               loc='lower right', bbox_to_anchor=(1.0, 1.0))   # di atas area plot, tidak menutupi batang
plt.tight_layout(); plt.show()

---
# Bagian A2 — Test-time augmentation (TTA) flip

Model yang sama dievaluasi dua kali di `my_val`: tanpa TTA dan dengan TTA. Dengan TTA, setiap gambar diprediksi dua kali (asli dan dicerminkan), lalu logits-nya dirata-ratakan.

TTA tidak mengubah bobot sama sekali, hanya cara prediksi. Biayanya: evaluasi menjadi 2× lebih lama. Itu sebabnya batas 15 menit dicek lagi di Bagian C.

In [ ]:
RUN_A = f'runs/A_{CHOSEN}'


def eval_copy(src_sub, tag, tta):
    d = os.path.join('eval_tmp', tag)
    shutil.rmtree(d, ignore_errors=True)
    shutil.copytree(src_sub, d)
    cfg = json.load(open(f'{d}/config.json'))
    cfg['tta_hflip'] = tta
    json.dump(cfg, open(f'{d}/config.json', 'w'), indent=2)
    run(f'python -u evaluate.py --submission {d} --data my_val --out {d}/hasil.json')
    return json.load(open(f'{d}/hasil.json'))


r_no = eval_copy(f'{RUN_A}/submission', 'tanpa_tta', False)
r_tta = eval_copy(f'{RUN_A}/submission', 'dengan_tta', True)
cmp_tab = pd.DataFrame({
    'tanpa TTA': [r_no['macro_f1'], r_no['accuracy']] + [r_no['per_class_f1'][c] for c in CLASSES],
    'dengan TTA': [r_tta['macro_f1'], r_tta['accuracy']] + [r_tta['per_class_f1'][c] for c in CLASSES],
}, index=['Macro-F1', 'akurasi'] + [f'F1 {c}' for c in CLASSES])
cmp_tab['selisih'] = cmp_tab['dengan TTA'] - cmp_tab['tanpa TTA']
display(cmp_tab.round(4))

if USE_TTA == 'auto':
    USE_TTA = r_tta['macro_f1'] >= r_no['macro_f1']
    print(f"USE_TTA = {USE_TTA}  (aturan: aktif bila Macro-F1 dengan TTA >= tanpa TTA; "
          f"{fmt(r_tta['macro_f1'])} vs {fmt(r_no['macro_f1'])})")
else:
    print('USE_TTA dipaksa =', USE_TTA)
F1_LOKAL = r_tta['macro_f1'] if USE_TTA else r_no['macro_f1']

### Galeri kesalahan (model terpilih)
Sama seperti Preview 1: urut dari yang paling **yakin salah**. Bandingkan dengan galeri Preview 1. Apakah keluarga gelas masih mendominasi?

In [ ]:
from math import ceil
from PIL import Image

pred = pd.read_csv(f'{RUN_A}/val_predictions.csv')
if src is not None:
    pred['title'] = pred.file.map(src.set_index('file')['title'])
err = pred[pred.benar == 0]
print(f'{len(err)} salah dari {len(pred)} gambar my_val (tanpa TTA)')
show = err.head(12)
if len(show):
    cols = 4
    nrows = ceil(len(show) / cols)
    fig, axes = plt.subplots(nrows, cols, figsize=(4 * cols, 4.4 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, show.iterrows()):
        ax.imshow(Image.open(os.path.join('my_val', r.file)).convert('RGB'))
        ax.set_title(f'label: {r.label}\nprediksi: {r.pred} ({r.p_pred:.2f})', fontsize=9, color=INK)
    plt.tight_layout(); plt.show()
    pd.set_option('display.max_colwidth', 80)
    display(show)

---
# Bagian B — Final: latih di seluruh data

Konfigurasi terpilih dilatih ulang dengan `my_train` + `my_val` (seluruh 2.964 gambar).

**Koreksi dari Preview 1:** jumlah epoch sekarang memakai **jadwal penuh** (`EPOCHS`), tidak lagi "epoch terbaik" dari Bagian A. Kurva Preview 1 sudah datar sejak epoch 9 (selisih antar-epoch hanya ±1 gambar), jadi "epoch terbaik" sebagian besar hanyalah noise. Selain itu, jadwal cosine untuk 12 epoch tidak sama dengan epoch ke-12 dari jadwal 15 epoch. Memakai jadwal yang sama persis dengan Bagian A adalah pilihan yang paling bisa dipertanggungjawabkan.

In [ ]:
FINAL_ARGS = EXPERIMENTS[CHOSEN] + ' --full' + (' --tta-hflip' if USE_TTA else '')
FINAL_NAME = f'B_{CHOSEN}_full' + ('_tta' if USE_TTA else '')
print('konfigurasi final:', FINAL_NAME, '|', COMMON, FINAL_ARGS)
restore_or_train(FINAL_NAME, FINAL_ARGS)
FINAL_SUB = f'runs/{FINAL_NAME}/submission'

---
# Bagian C — Paket submission + README

1. **Cek batas waktu.** Evaluasi dijalankan **di CPU** (GPU disembunyikan) pada 100 gambar, lalu diekstrapolasi ke 740 gambar. Batas dari aturan adalah 15 menit. Kita tidak tahu apakah asisten memakai GPU, jadi kita asumsikan kondisi terburuk.
2. **Paket.** Folder `submission/` (4 file) dan `README.MD` yang sudah terisi dengan hasil notebook ini. Paket diuji lagi dengan `evaluate.py` resmi.

In [ ]:
probe = 'cpu_probe'
shutil.rmtree(probe, ignore_errors=True)
for c in CLASSES:
    os.makedirs(f'{probe}/{c}')
    for f in sorted(os.listdir(f'my_val/{c}'))[:20]:
        os.symlink(os.path.abspath(f'my_val/{c}/{f}'), f'{probe}/{c}/{f}')
run(f'CUDA_VISIBLE_DEVICES= python -u evaluate.py --submission {FINAL_SUB} --data {probe} --out cpu_probe.json')
cpu = json.load(open('cpu_probe.json'))
CPU_MIN = cpu['seconds'] / cpu['n_images'] * 740 / 60
print(f'\nEstimasi evaluasi 740 gambar di CPU Colab ({os.cpu_count()} core): {CPU_MIN:.1f} menit (batas 15 menit)')
if CPU_MIN > 10:
    print('PERINGATAN: margin tipis. Pertimbangkan USE_TTA = False (evaluasi 2x lebih cepat), lalu jalankan ulang Bagian B.')

In [ ]:
PAKET = 'paket_preview2'
shutil.rmtree(PAKET, ignore_errors=True)
dst = os.path.join(PAKET, 'submission')
os.makedirs(dst)
for f in ('model.py', 'config.json', 'model.pth'):
    shutil.copy2(os.path.join(FINAL_SUB, f), dst)
Path(dst, 'requirements.txt').write_text(f'timm=={timm.__version__}\n')
print('===== cek paket dengan evaluate.py resmi =====')
run(f'python -u evaluate.py --submission {dst} --data my_val')
print('(model B ikut dilatih dengan my_val -> skor di atas hanya membuktikan paket bisa dimuat.)')
shutil.rmtree(os.path.join(dst, '__pycache__'), ignore_errors=True)   # cache Python, bukan bagian submission

# ----------------------------------------------------------------- README.MD
fcfg = json.load(open(os.path.join(dst, 'config.json')))
fbest = json.load(open(f'runs/{FINAL_NAME}/best.json'))
img = fcfg['img_size']
BACKBONE_DESC = {'convnext_tiny.fb_in22k_ft_in1k': 'ConvNeXt-Tiny pretrained ImageNet-22k → fine-tuned ImageNet-1k'}
backbone_desc = BACKBONE_DESC.get(fcfg['backbone'], 'backbone pretrained ImageNet dari timm')
aug_text = (f'RandomResizedCrop({img}, skala 0,4–1,0), flip horizontal, TrivialAugmentWide, RandomErasing(p=0,25)'
            if AUG == 'strong' else f'RandomResizedCrop({img}, skala 0,6–1,0), flip horizontal, ColorJitter(0,2)')
tta_text = ('**aktif**: prediksi = rata-rata logits gambar asli dan cerminan horizontalnya (`tta_hflip` di '
            f'`config.json`). Di `my_val`: {fmt(r_no["macro_f1"])} tanpa TTA → {fmt(r_tta["macro_f1"])} dengan TTA.'
            if USE_TTA else
            f'tidak dipakai (di `my_val`: {fmt(r_no["macro_f1"])} tanpa TTA vs {fmt(r_tta["macro_f1"])} dengan TTA).')
exp_rows = []
for k, name in enumerate(EXPERIMENTS):
    s, b = summary.loc[name], bests[name]
    weakest = min(b['per_class_f1'], key=b['per_class_f1'].get)
    exp_rows.append(
        f"| E{k + 2} | {name}: `{EXPERIMENTS[name]}`, data lengkap, {EPOCHS} epoch | {fmt(s.f1_terbaik)} (epoch {int(s.epoch_terbaik)}); "
        f"rata-3-akhir {fmt(s.f1_rata3_akhir)} | F1 gelas {fmt(s.gelas, 3)}, kertas {fmt(s.kertas, 3)}, styrofoam {fmt(s.styrofoam, 3)}; "
        f"terlemah `{weakest}`{' — **terpilih**' if name == CHOSEN else ''} |")
k_tta = len(EXPERIMENTS) + 2
exp_rows.append(f"| E{k_tta} | TTA flip pada {CHOSEN} | {fmt(r_no['macro_f1'])} → {fmt(r_tta['macro_f1'])} | "
                f"{'dipakai' if USE_TTA else 'tidak dipakai'} |")
exp_rows.append(f"| P2 | {CHOSEN} dilatih ulang di seluruh data ({fbest['n_train']} gambar), {EPOCHS} epoch"
                f"{', TTA flip' if USE_TTA else ''} | — (tanpa validasi; konfigurasi sama dengan eksperimen {CHOSEN}) | dikirim untuk Preview 2 |")
repro_a = f'python train.py --data ../data/train --split-dir . --name A_{CHOSEN} {COMMON} {EXPERIMENTS[CHOSEN]}'
repro_b = f'python train.py --data ../data/train --split-dir . --name {FINAL_NAME} {COMMON} {FINAL_ARGS}'

README = string.Template('''# Studio 2 — Klasifikasi Sampah · KELOMPOK 1

Deep Learning (IF25-40401) · Teknik Informatika ITERA · Semester Ganjil 2026/2027

## Anggota Kelompok

| No | Nama Lengkap | NIM | Kontribusi utama |
|----|--------------|-----|------------------|
| 1  | Ahmat Prayoga Sembiring | 123140053 |  |
| 2  | Tengku Hafid Diraputra | 123140043 |  |

## Ringkasan Pendekatan

- **Arsitektur / backbone**: $backbone_desc (`timm`: `$backbone`), head linear 5 kelas. Seluruh layer di-*fine-tune* (tanpa freezing).
- **Strategi pelatihan**:
  - AdamW, LR backbone $lr, LR head 10×, weight decay 0,05; warmup linear 1 epoch lalu cosine decay per-step; $epochs epoch; batch $batch; mixed precision (AMP)
  - Cross-entropy dengan *label smoothing* 0,1 (membantu menghadapi label noise)
  - Augmentasi: $aug_text
- **Seleksi model**: eksperimen dibandingkan pada split validasi lokal yang sama (15% per kelas, seed 42) dengan Macro-F1 dari fungsi `metrics()` di `evaluate.py` resmi. Pemilihan memakai **rata-rata Macro-F1 3 epoch terakhir**, bukan nilai maksimum, untuk menghindari bias seleksi pada validasi yang kecil. Model submission lalu dilatih ulang pada **seluruh data** ($n_total gambar) dengan konfigurasi dan jadwal yang sama.
- **Data tambahan**: tidak ada. Hanya train set resmi (lengkap, diverifikasi terhadap `train_sources.csv`).
- **Ukuran input & preprocessing**: resize sisi pendek $resize → center crop $img → normalisasi mean/std ImageNet. Disimpan di `submission/config.json` dan dibaca oleh `get_transform()`.
- **Test-time augmentation**: $tta_text
- **Bobot**: fp16 ($mb MB) agar di bawah batas 100 MB GitHub; argmax identik dengan fp32 (diuji).

## Hasil

| Putaran | Commit | Macro-F1 (split validasi lokal) | Macro-F1 (diumumkan asisten) |
|---------|--------|----------------------------------|------------------------------|
| Preview 1 (14 Okt) | `$c1` | 0,9722 (split lama, data belum lengkap) | $s1 |
| Preview 2 (21 Okt) | `<isi hash setelah push>` | $f1_local ($chosen$tta_short, model A) | |
| Final (25 Okt)     | | | |

Split validasi lokal Preview 2 berbeda dari Preview 1 karena dataset sudah lengkap, sehingga kedua angka lokal tidak bisa dibandingkan langsung. Pembanding yang adil adalah skor dari asisten, karena dihitung pada validation set tersembunyi yang sama.

Catatan eksperimen:

| ID | Konfigurasi | Macro-F1 lokal | Catatan |
|----|-------------|----------------|---------|
| E0 | Baseline ResNet-18 (starter kit, 8 epoch) | — | referensi: ~0,92 di validation set tersembunyi (halaman kompetisi) |
| E1 | ConvNeXt-Tiny 224, 15 epoch, aug strong, **2.753 gambar** (salinan dataset belum lengkap) | 0,9722 (epoch 12) | Model Preview 1. Kesalahan dominan: keluarga gelas (plastik/kertas/styrofoam), bentuk sama tetapi material berbeda → hipotesis: resolusi lebih tinggi membantu membedakan tekstur |
$exp_rows

## Cara Reproduksi

Dataset tidak disertakan di repo (sesuai aturan kompetisi). Letakkan folder `train/` dari Google Drive asisten di luar repo.

**Google Colab (yang dipakai kelompok):** buka `notebooks/studio2_preview2.ipynb`, pilih runtime T4 GPU, isi `DATA_SRC`, lalu *Run all*. Notebook menjalankan semua eksperimen, uji TTA, model final, dan membuat paket submission.

**Lokal (setara dengan notebook):**

```bash
pip install -r requirements.txt
# eksperimen dengan validasi (my_val dibuat otomatis, seed 42)
$repro_a
# model submission: data penuh, jadwal yang sama
$repro_b
# salin isi runs/$final_name/submission/ ke submission/, lalu:
python evaluate.py --submission ./submission --data ./my_val
```

## Struktur Repositori

```
submission/
  model.py          # get_model() dan get_transform() — membaca config.json (termasuk tta_hflip)
  config.json       # backbone + ukuran input + mean/std + tta_hflip
  model.pth         # state_dict (fp16, $mb MB)
  requirements.txt  # timm==$timm_ver
evaluate.py         # salinan skrip evaluasi resmi (tidak diubah)
train.py            # training: seleksi Macro-F1, log, ekspor submission, --full, --tta-hflip
train_baseline.py   # baseline dari starter kit
notebooks/          # notebook Colab versi bersih (tanpa output)
```
''').safe_substitute(
    backbone=fcfg['backbone'], backbone_desc=backbone_desc, lr=f'{LR:g}', epochs=EPOCHS, batch=BATCH, aug_text=aug_text, n_total=fbest['n_train'],
    resize=int(round(img / fcfg['crop_pct'])), img=img, tta_text=tta_text, mb=fmt(fbest['model_mb'], 1),
    c1=COMMIT_PREVIEW1, s1=SKOR_PREVIEW1, f1_local=fmt(F1_LOKAL), chosen=CHOSEN,
    tta_short=' + TTA' if USE_TTA else '', exp_rows='\n'.join(exp_rows), repro_a=repro_a, repro_b=repro_b,
    final_name=FINAL_NAME, timm_ver=timm.__version__)
Path(PAKET, 'README.MD').write_text(README)
print('README.MD dibuat ->', os.path.join(PAKET, 'README.MD'))

ZIP = shutil.make_archive('paket_preview2', 'zip', PAKET)
print(f'\n{ZIP}  ({os.path.getsize(ZIP) / 1e6:.1f} MB)  berisi: submission/ (4 file) + README.MD')
if SAVE_TO_DRIVE:
    os.makedirs(SAVE_TO_DRIVE, exist_ok=True)
    shutil.copy2(ZIP, os.path.join(SAVE_TO_DRIVE, f'paket_preview2_{FINAL_NAME}_{time.strftime("%m%d-%H%M")}.zip'))
    print('salinan zip disimpan di Drive:', SAVE_TO_DRIVE)
if IN_COLAB:
    from google.colab import files
    files.download(ZIP)

print('\nSaran pesan commit:')
print(f'git commit -m "Preview 2: {CHOSEN}{" + TTA flip" if USE_TTA else ""}, data lengkap {fbest["n_train"]} gambar, '
      f'Macro-F1 lokal {F1_LOKAL:.4f}"')

---
## Setelah notebook selesai (di laptop)

`paket_preview2.zip` terunduh otomatis dan salinannya ada di Drive (`studio2_runs/`). Isinya: `submission/` (4 file) + `README.MD`.

1. Ekstrak zip, lalu **timpa** folder `submission/` dan file `README.MD` di repo.
2. Cek `submission/config.json`: nilai `img_size` dan `tta_hflip` harus sesuai pilihan notebook.
3. Salin juga file kode versi baru ke repo (dari paket kode yang diberikan bersama notebook ini): `train.py` dan `notebooks/studio2_preview2.ipynb` **versi bersih**, bukan yang sudah dijalankan.
4. `git add --dry-run .` → pastikan tidak ada `.jpg` atau `.zip`.
5. Commit dan push **sebelum Rabu, 21 Okt 23:59**. Pesan commit yang disarankan ada di output sel sebelumnya.
6. Setelah push, ganti `<isi hash setelah push>` di README dengan hash commit tadi (`git log --oneline -1`), lalu commit lagi.